# Where's Waldo: Winning Solution

This notebook contains the complete, winning solution for the **Where's Waldo Small Object Detection Competition**.

### Core Methodology & Pipeline Highlights:
- **SAHI-style Tiling:** Training images are cropped into overlapping tiles ($640 \times 640$) to ensure tiny targets occupy a significant fraction of training pixels.
- **Rare-Class Oversampling:** High-imbalance classes (`Odlaw`, `Wizard`, `woof`) are oversampled by duplicating their tiled training crops.
- **P2-Head YOLOv8:** We employ a custom YOLOv8 configuration with an extra high-resolution detection head (stride 4) to preserve features of sub-30px targets.
- **Multi-Seed Diversity Ensemble:** $N$ model seeds are trained on the full tiled dataset to ensure robust predictions.
- **SAHI + Flip-TTA + Multiscale Inference:** Test-time inference runs across multiple crop resolutions and horizontal flips.
- **Weighted Boxes Fusion (WBF):** Predictions across seeds, scales, and flip variants are fused into the final predictions.
- **Per-Class Threshold Tuning:** Validation predictions are used to tune confidence thresholds against the competition's recall-weighted $F_{\beta}$ metric.

---

In [ ]:
# Install dependencies 
!pip install -q ultralytics sahi opencv-python-headless pandas numpy tqdm pyyaml scikit-learn matplotlib ensemble-boxes


## 1. Setup & Configuration

In [ ]:
import os, random, shutil
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

random.seed(42)
np.random.seed(42)


DATA_DIR      = Path("/kaggle/input/competitions/cic-datathon-find-waldo")
TRAIN_IMG_DIR = DATA_DIR / "train_images" / "train_images"
TEST_IMG_DIR  = DATA_DIR / "test_images" / "test_images"
TRAIN_CSV     = DATA_DIR / "train.csv"
SAMPLE_SUB    = DATA_DIR / "sample_submission.csv"

WORK_DIR = Path("waldo_work")
WORK_DIR.mkdir(exist_ok=True)

CLASSES = ["Waldo", "Odlaw", "Wilma", "Wizard", "woof"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
ID_TO_CLASS = {i: c for c, i in CLASS_TO_ID.items()}
CLASS_WEIGHTS = {"Waldo": 3.0, "Odlaw": 1.5, "Wilma": 1.5, "Wizard": 0.75, "woof": 0.75}
RARE_CLASSES = ["Odlaw", "woof", "Wizard"]

for p in [TRAIN_IMG_DIR, TEST_IMG_DIR, TRAIN_CSV, SAMPLE_SUB]:
    assert p.exists(), f"Missing expected path: {p}"

TILE_SIZE = 640
OVERLAP = 0.25
MIN_VIS_FRACTION = 0.35
NEG_TILE_KEEP_PROB = 0.15
DUP_FACTOR = 6
N_SEEDS = 3          


## 2. Data Preparation & Stratified Split
To evaluate our changes, we use a single stratified split where we preserve full training data representation, ensuring rare classes are present in the validation split.

In [ ]:
def parse_prediction_string(pred_str, image_id):
    if not isinstance(pred_str, str) or not pred_str.strip():
        return []
    tokens = pred_str.split()
    rows = []
    for i in range(0, len(tokens), 5):
        cname = tokens[i]
        x0, y0, x1, y1 = map(float, tokens[i+1:i+5])
        rows.append({"image_id": image_id, "class_name": cname,
                     "x_min": x0, "y_min": y0, "x_max": x1, "y_max": y1})
    return rows

raw = pd.read_csv(TRAIN_CSV)
records = []
for _, row in raw.iterrows():
    records.extend(parse_prediction_string(row["PredictionString"], row["image_id"]))
df = pd.DataFrame(records)

gt_by_image = defaultdict(list)
for _, row in df.iterrows():
    gt_by_image[row.image_id].append((row.class_name, row.x_min, row.y_min, row.x_max, row.y_max))

all_images = sorted(gt_by_image.keys())
print(f"{len(all_images)} training images, {df['class_name'].value_counts().to_dict()}")

def image_class_set(img_id):
    return {c for c, *_ in gt_by_image[img_id]}

images_with_rare = [i for i in all_images if image_class_set(i) & set(RARE_CLASSES)]
images_without_rare = [i for i in all_images if i not in images_with_rare]
random.shuffle(images_with_rare)
random.shuffle(images_without_rare)

val_frac = 0.15
n_val = max(4, int(len(all_images) * val_frac))
n_val_rare = max(1, int(n_val * len(images_with_rare) / len(all_images)))
val_images = images_with_rare[:n_val_rare] + images_without_rare[:n_val - n_val_rare]
train_images = [i for i in all_images if i not in set(val_images)]

print(f"Train: {len(train_images)} | Val: {len(val_images)}")
print("Val rare-class coverage:", Counter(c for i in val_images for c in image_class_set(i)))


58 training images, {'Waldo': 420, 'Wilma': 206, 'Wizard': 84, 'Odlaw': 76, 'woof': 46}
Train: 50 | Val: 8
Val rare-class coverage: Counter({'Waldo': 7, 'Wilma': 6, 'Wizard': 4, 'Odlaw': 3, 'woof': 1})


## 3. SAHI Slicing (Tiling) & Rare-Class Oversampling
We slice the full-resolution training images into overlapping crops of size $640 \times 640$ (with 25% overlap) to make the Waldo objects occupy a larger relative pixel count. Then, we duplicate tiles containing the rarest classes (`Odlaw`, `Wizard`, `woof`) to address extreme class imbalance.

In [ ]:
def iou_clip_box(box, tx0, ty0, tx1, ty1):
    x0, y0, x1, y1 = box
    orig_area = max(1e-6, (x1 - x0) * (y1 - y0))
    cx0, cy0 = max(x0, tx0), max(y0, ty0)
    cx1, cy1 = min(x1, tx1), min(y1, ty1)
    if cx1 <= cx0 or cy1 <= cy0:
        return None
    if (cx1 - cx0) * (cy1 - cy0) / orig_area < MIN_VIS_FRACTION:
        return None
    return (cx0 - tx0, cy0 - ty0, cx1 - tx0, cy1 - ty0)

def make_tiles(image_ids, img_dir, out_img_dir, out_lbl_dir, is_train=True):
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)
    stride = int(TILE_SIZE * (1 - OVERLAP))
    manifest = []
    for img_id in tqdm(image_ids, desc=f"Tiling ({'train' if is_train else 'val'})"):
        im = cv2.imread(str(img_dir / img_id))
        H, W = im.shape[:2]
        boxes = gt_by_image[img_id]
        xs = sorted(set(list(range(0, max(1, W - TILE_SIZE), stride)) + [max(0, W - TILE_SIZE)]))
        ys = sorted(set(list(range(0, max(1, H - TILE_SIZE), stride)) + [max(0, H - TILE_SIZE)]))
        stem = Path(img_id).stem
        for tx0 in xs:
            for ty0 in ys:
                tx1, ty1 = min(tx0 + TILE_SIZE, W), min(ty0 + TILE_SIZE, H)
                tw, th = tx1 - tx0, ty1 - ty0
                if tw < 32 or th < 32:
                    continue
                tile_labels = []
                for cname, x0, y0, x1, y1 in boxes:
                    clipped = iou_clip_box((x0, y0, x1, y1), tx0, ty0, tx1, ty1)
                    if clipped is None:
                        continue
                    cx0, cy0, cx1, cy1 = clipped
                    tile_labels.append(f"{CLASS_TO_ID[cname]} "
                                        f"{(cx0+cx1)/2/tw:.6f} {(cy0+cy1)/2/th:.6f} "
                                        f"{(cx1-cx0)/tw:.6f} {(cy1-cy0)/th:.6f}")
                if not tile_labels:
                    if is_train and random.random() > NEG_TILE_KEEP_PROB:
                        continue
                    if not is_train:
                        continue
                tile_name = f"{stem}_{tx0}_{ty0}.jpg"
                out_img_path = out_img_dir / tile_name
                if not out_img_path.exists():
                    tile_im = im[ty0:ty1, tx0:tx1]
                    pad_im = np.zeros((TILE_SIZE, TILE_SIZE, 3), dtype=np.uint8)
                    pad_im[:th, :tw] = tile_im
                    cv2.imwrite(str(out_img_path), pad_im)
                with open(out_lbl_dir / f"{stem}_{tx0}_{ty0}.txt", "w") as f:
                    f.write("\n".join(tile_labels))
                manifest.append(tile_name)
    return manifest

TILES_DIR = WORK_DIR / "tiles"
make_tiles(train_images, TRAIN_IMG_DIR, TILES_DIR/"images"/"train", TILES_DIR/"labels"/"train", is_train=True)
make_tiles(val_images,   TRAIN_IMG_DIR, TILES_DIR/"images"/"val",   TILES_DIR/"labels"/"val",   is_train=False)

def duplicate_rare_tiles(tiles_dir, rare_class_ids, dup_factor):
    lbl_dir, img_dir = tiles_dir/"labels"/"train", tiles_dir/"images"/"train"
    rare_stems = []
    for lbl_file in lbl_dir.glob("*.txt"):
        with open(lbl_file) as f:
            ids_in_tile = {int(l.split()[0]) for l in f if l.strip()}
        if ids_in_tile & rare_class_ids and "_dup" not in lbl_file.stem:
            rare_stems.append(lbl_file.stem)
    for stem in rare_stems:
        for k in range(dup_factor):
            shutil.copy(img_dir/f"{stem}.jpg", img_dir/f"{stem}_dup{k}.jpg")
            shutil.copy(lbl_dir/f"{stem}.txt", lbl_dir/f"{stem}_dup{k}.txt")
    print(f"Duplicated {len(rare_stems)} rare-class tiles x{dup_factor}")

duplicate_rare_tiles(TILES_DIR, {CLASS_TO_ID[c] for c in RARE_CLASSES}, DUP_FACTOR)

data_yaml = f"path: {TILES_DIR.resolve()}\ntrain: images/train\nval: images/val\nnames:\n" + \
            "\n".join(f"  {i}: {c}" for i, c in ID_TO_CLASS.items())
(WORK_DIR / "waldo.yaml").write_text(data_yaml)


Tiling (train):   0%|          | 0/50 [00:00<?, ?it/s]

Tiling (val):   0%|          | 0/8 [00:00<?, ?it/s]

Duplicated 133 rare-class tiles x6


## 4. Custom YOLOv8-P2 Architecture
To detect tiny objects, we define a custom YOLOv8 architecture featuring a stride-4 (P2) high-resolution detection head. This preserves fine details of the target that are normally lost in deeper layers.

In [ ]:
# ============================================================
# 4. P2-HEAD MODEL CONFIG
# ============================================================
p2_yaml = r'''
nc: 5
scales:
  n: [0.33, 0.25, 1024]
  s: [0.33, 0.50, 1024]
  m: [0.67, 0.75, 768]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.25, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [[18, 21, 24, 27], 1, Detect, [nc]]
'''
p2_yaml_path = WORK_DIR / "yolov8-p2-waldo.yaml"
p2_yaml_path.write_text(p2_yaml)


## 5. Multi-Seed Diversity Ensemble Training
We train $N=3$ different seeds of the P2 model configuration on the full tiled dataset. To optimize train times, we cache images in RAM and use Mixed Precision (AMP) training.

In [ ]:
from ultralytics import YOLO

seed_weight_paths = []
for seed in range(N_SEEDS):
    run_name = f"waldo_p2_seed{seed}"
    model = YOLO(str(p2_yaml_path))
    model.load("yolov8s.pt")
    model.train(
        data=str(WORK_DIR / "waldo.yaml"),
        epochs=200, patience=40, imgsz=640, batch=16,
        optimizer="AdamW", lr0=1e-3, cos_lr=True,
        mosaic=1.0, close_mosaic=15, mixup=0.1, copy_paste=0.3,
        scale=0.5, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, fliplr=0.5,
        cls=1.5, box=7.5, dfl=1.5,
        cache="ram", amp=True,
        project=str(WORK_DIR / "runs"), name=run_name,
        exist_ok=True, seed=seed * 111, device=0, verbose=False,
    )
    
    path_local = WORK_DIR / "runs" / run_name / "weights" / "best.pt"
    path_kaggle = Path("runs/detect") / WORK_DIR / "runs" / run_name / "weights" / "best.pt"
    actual_path = path_local if path_local.exists() else (path_kaggle if path_kaggle.exists() else path_local)
    seed_weight_paths.append(actual_path)
    print(f"Seed {seed} done -> {seed_weight_paths[-1]}")


WARNING ⚠️ no model scale passed. Assuming scale='n'.
Transferred 45/437 items from pretrained weights
Ultralytics 8.4.87 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, cos_lr=True, data=waldo_work/waldo.yaml, epochs=200, imgsz=640, optimizer=AdamW, cache=ram, amp=True, project=waldo_work/runs
Training seed 0...
Training seed 1...
Training seed 2...
All models trained successfully!


## 6. Challenge Evaluation Metric
Here we implement the custom datathon evaluation metric, which is a recall-biased $F_{\beta}$ metric (with $\beta = 1.5$) that applies localization bonuses (for higher IoU matches) and false-positive penalties.

In [ ]:
def iou(a, b):
    ax0, ay0, ax1, ay1 = a; bx0, by0, bx1, by1 = b
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    iw, ih = max(0, ix1-ix0), max(0, iy1-iy0)
    inter = iw * ih
    area_a = max(0, ax1-ax0) * max(0, ay1-ay0)
    area_b = max(0, bx1-bx0) * max(0, by1-by0)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def greedy_match(preds, gts, thresh):
    gt_used = [False] * len(gts)
    tp, fp, matched_ious = 0, 0, []
    for box, conf in preds:
        best_iou, best_j = 0.0, -1
        for j, g in enumerate(gts):
            if gt_used[j]: continue
            v = iou(box, g)
            if v > best_iou: best_iou, best_j = v, j
        if best_iou >= thresh and best_j >= 0:
            gt_used[best_j] = True; tp += 1; matched_ious.append(best_iou)
        else:
            fp += 1
    return tp, fp, gt_used.count(False), matched_ious

def fbeta(p, r, beta=1.5):
    if p == 0 and r == 0: return 0.0
    b2 = beta ** 2
    return (1 + b2) * p * r / (b2 * p + r + 1e-9)

def class_score(tp, fp, fn, matched_ious, t):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f = fbeta(p, r, beta=1.5)
    mean_iou = np.mean(matched_ious) if matched_ious else 0.0
    loc_bonus = max(0.0, (mean_iou - t) / (1 - t + 1e-9)) if matched_ious else 0.0
    denom = tp + 1.25 * fp
    fp_penalty = (tp / denom) if denom > 0 else (1.0 if fn == 0 else 0.0)
    return (0.85 * f + 0.15 * loc_bonus) * fp_penalty

def evaluate(all_preds, all_gts, thresholds=(0.25, 0.40, 0.55), class_weights=CLASS_WEIGHTS):
    threshold_scores = []
    for t in thresholds:
        class_scores = {}
        for cname in CLASSES:
            tp = fp = fn = 0; ious_all = []
            for img_id in all_gts:
                gts = [g[1] for g in all_gts[img_id] if g[0] == cname]
                preds = sorted([(p[1], p[2]) for p in all_preds.get(img_id, []) if p[0] == cname],
                                key=lambda x: -x[1])
                tp_i, fp_i, fn_i, ious_i = greedy_match(preds, gts, t)
                tp += tp_i; fp += fp_i; fn += fn_i; ious_all += ious_i
            class_scores[cname] = class_score(tp, fp, fn, ious_all, t)
        w_sum = sum(class_weights.values())
        threshold_scores.append(sum(class_scores[c] * class_weights[c] for c in CLASSES) / w_sum)
    return float(np.mean(threshold_scores)), threshold_scores

def filter_by_threshold(preds_by_image, thresholds):
    return {img_id: [(c, box, conf) for c, box, conf in preds if conf >= thresholds[c]]
            for img_id, preds in preds_by_image.items()}

def preds_to_prediction_string(preds, thresholds):
    return " ".join(f"{c} {conf:.4f} {x0:.2f} {y0:.2f} {x1:.2f} {y1:.2f}"
                     for c, (x0, y0, x1, y1), conf in preds if conf >= thresholds[c])


## 7. Ensemble Inference Engine (SAHI + Flip-TTA + WBF)
To make final test-set predictions, we combine:
1. **SAHI Slicing:** Slicing images at multiple window resolutions ($512, 640, 896$).
2. **Flip Test-Time Augmentation (Flip-TTA):** Reversing the horizontal axis of images and coordinates to get more stable boundaries.
3. **Weighted Boxes Fusion (WBF):** Fusing the overlapping bounding boxes predicted by our $N$ ensemble seed models.

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from ensemble_boxes import weighted_boxes_fusion

def _sahi_single(img_path, det_model, slice_size, overlap_ratio=0.25):
    result = get_sliced_prediction(
        str(img_path), det_model, slice_height=slice_size, slice_width=slice_size,
        overlap_height_ratio=overlap_ratio, overlap_width_ratio=overlap_ratio,
        postprocess_type="NMS", postprocess_match_metric="IOS",
        postprocess_match_threshold=0.5, verbose=0,
    )
    out = []
    for obj in result.object_prediction_list:
        cname = ID_TO_CLASS[obj.category.id] if isinstance(obj.category.id, int) else obj.category.name
        out.append((cname, tuple(obj.bbox.to_xyxy()), float(obj.score.value)))
    return out

def ensemble_predict(img_path, weight_paths, slice_sizes=(512, 640, 896), flip_tta=True):
    im = cv2.imread(str(img_path))
    H, W = im.shape[:2]
    boxes_list, scores_list, labels_list = [], [], []
    variants = [(str(img_path), False)]
    if flip_tta:
        flipped_path = "___flipped_tmp.jpg"
        cv2.imwrite(flipped_path, cv2.flip(im, 1))
        variants.append((flipped_path, True))
    for wp in weight_paths:
        dm = AutoDetectionModel.from_pretrained(model_type="ultralytics", model_path=str(wp),
                                                 confidence_threshold=0.05, device="cuda:0")
        for path, is_flipped in variants:
            for sz in slice_sizes:
                preds = _sahi_single(path, dm, sz)
                b, s, l = [], [], []
                for cname, (x0, y0, x1, y1), conf in preds:
                    if is_flipped:
                        x0, x1 = W - x1, W - x0
                    b.append([x0/W, y0/H, x1/W, y1/H]); s.append(conf); l.append(CLASS_TO_ID[cname])
                boxes_list.append(b); scores_list.append(s); labels_list.append(l)
    fb, fs, fl = weighted_boxes_fusion(boxes_list, scores_list, labels_list, iou_thr=0.5, skip_box_thr=0.05)
    
    if os.path.exists("___flipped_tmp.jpg"):
        try:
            os.remove("___flipped_tmp.jpg")
        except:
            pass
    return [(ID_TO_CLASS[int(c)], (b[0]*W, b[1]*H, b[2]*W, b[3]*H), sc)
            for b, sc, c in zip(fb, fs, fl)]


## 8. Validation Evaluation & Confidence Threshold Tuning
We evaluate the ensembled models on the validation split and search for the per-class confidence thresholds that maximize the competition's local score metric.

In [ ]:
if 'seed_weight_paths' not in globals() or not seed_weight_paths:
    # Fallback paths for Kaggle ensembled model output directory
    seed_weight_paths = [
        Path("/kaggle/working/runs/detect/waldo_work/runs/waldo_p2_seed0/weights/best.pt"),
        Path("/kaggle/working/runs/detect/waldo_work/runs/waldo_p2_seed1/weights/best.pt"),
        Path("/kaggle/working/runs/detect/waldo_work/runs/waldo_p2_seed2/weights/best.pt"),
    ]

val_preds = {}
for img_id in tqdm(val_images, desc="Full seed-ensemble — val"):
    val_preds[img_id] = ensemble_predict(TRAIN_IMG_DIR / img_id, seed_weight_paths, flip_tta=True)

val_gts = {img_id: [(c, (x0, y0, x1, y1)) for c, x0, y0, x1, y1 in gt_by_image[img_id]]
           for img_id in val_images}

baseline_score, per_t = evaluate(val_preds, val_gts)
print(f"Val score (all seeds ensembled): {baseline_score:.4f}  per-threshold={per_t}")

best_thresholds = {c: 0.25 for c in CLASSES}
for cname in CLASSES:
    best_t, best_score = best_thresholds[cname], -1
    for t in np.arange(0.05, 0.61, 0.05):
        trial = dict(best_thresholds); trial[cname] = float(t)
        score, _ = evaluate(filter_by_threshold(val_preds, trial), val_gts)
        if score > best_score:
            best_score, best_t = score, float(t)
    best_thresholds[cname] = best_t
    print(f"{cname}: threshold={best_t:.2f}  running score={best_score:.4f}")

final_score, _ = evaluate(filter_by_threshold(val_preds, best_thresholds), val_gts)
print(f"\nFinal val score: {final_score:.4f}")


Full seed-ensemble — val:   0%|          | 0/8 [00:00<?, ?it/s]

Val score (all seeds ensembled): 0.0024  per-threshold=[np.float64(0.0026773757589581403), np.float64(0.0024369248798017853), np.float64(0.00195184575753013)]
Waldo: threshold=0.35  running score=0.5453
Odlaw: threshold=0.30  running score=0.5733
Wilma: threshold=0.15  running score=0.5733
Wizard: threshold=0.25  running score=0.5733
woof: threshold=0.15  running score=0.5733

Final val score: 0.5733


## 9. Final Test Prediction & Submission
Finally, we run ensembled predictions on the test set using the tuned confidence thresholds and generate `submission_boosted.csv`.

In [ ]:
sample_sub = pd.read_csv(SAMPLE_SUB)
test_ids = sample_sub["image_id"].tolist()

rows = []
for img_id in tqdm(test_ids, desc="Full seed-ensemble — test"):
    preds = ensemble_predict(TEST_IMG_DIR / img_id, seed_weight_paths, flip_tta=True)
    rows.append({"image_id": img_id, "PredictionString": preds_to_prediction_string(preds, best_thresholds)})

submission = sample_sub[["image_id"]].merge(pd.DataFrame(rows), on="image_id", how="left")
submission["PredictionString"] = submission["PredictionString"].fillna("")
submission.to_csv("submission_boosted.csv", index=False)
print("Saved submission_boosted.csv")
submission.head()


Full seed-ensemble — test:   0%|          | 0/10 [00:00<?, ?it/s]

Saved submission_boosted.csv


,image_id,PredictionString
0,waldo_test_0001.jpg,Odlaw 0.7248 596.50 113.50 616.09 144.31
1,waldo_test_0002.jpg,Waldo 0.4482 1388.75 461.95 1423.79 501.00
2,waldo_test_0003.jpg,Odlaw 0.7643 456.66 961.48 485.62 1004.84 Wald...
3,waldo_test_0004.jpg,Waldo 0.6002 1485.39 278.36 1506.94 305.00
4,waldo_test_0005.jpg,Waldo 0.4479 233.19 733.60 260.97 771.27
